# Session 05: 모델 로딩 패턴 심화

## 학습 목표

이번 세션에서는 Session 04에서 만든 기본 API를 **실무 수준**으로 확장합니다.

### 다루는 내용
1. **Lifespan 이벤트** 심화 - 서버 시작/종료 시 리소스 관리
2. **배치 예측 API** (`/predict/batch`) - 여러 건을 한 번에 처리
3. **모델 메타 정보 API** (`/model/info`) - 운영 중 모델 상태 확인
4. **요청 추적** - UUID `request_id`와 `timestamp`를 응답에 추가

### WHY 이런 기능들이 필요한가?
- **배치 예측**: 실무에서 심사 시스템이 한 번에 수십~수백 건을 요청하는 경우가 많음
- **모델 정보**: "지금 어떤 모델이 서빙되고 있는지" 확인할 수 없으면 운영이 불가능
- **요청 추적**: 문제 발생 시 특정 요청을 추적할 수 있어야 디버깅 가능

In [ ]:
import json
import uuid
from datetime import datetime
from pathlib import Path

## 1. Lifespan 이벤트 심화

### Lifespan 패턴이란?

FastAPI의 `lifespan`은 **비동기 컨텍스트 매니저**로, 서버의 시작과 종료를 하나의 함수에서 관리합니다.

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    # ── yield 전: 서버 시작 시 실행 ──
    # 모델 로딩, DB 연결 등
    
    yield  # ← 서버가 요청을 처리하는 동안 여기서 대기
    
    # ── yield 후: 서버 종료 시 실행 ──
    # 리소스 정리, 연결 해제 등
```

### WHY `@app.on_event("startup")` 대신 lifespan인가?

| 비교 항목 | `on_event` (구식) | `lifespan` (권장) |
|-----------|------------------|-------------------|
| 상태 | Deprecated | 현재 표준 |
| 시작+종료 | 두 개의 함수 필요 | 하나의 함수로 관리 |
| 리소스 공유 | 글로벌 변수 사용 | yield 전후에서 자연스럽게 공유 |
| 에러 처리 | try-except 별도 작성 | with 문 패턴으로 자동 정리 |

In [ ]:
# 여기에 코드를 작성하세요


## 2. 배치 예측 API

### WHY 배치 엔드포인트를 만드는가?

| 방식 | 100건 처리 | 네트워크 오버헤드 |
|------|-----------|------------------|
| 단건 100번 호출 | 100번의 HTTP 왕복 | 높음 |
| 배치 1번 호출 | 1번의 HTTP 왕복 | 낮음 |

- HTTP 연결 설정/해제 비용이 크므로, 한 번에 모아서 보내는 것이 효율적
- 모델 추론도 내부적으로 벡터화할 수 있어 건별 호출보다 빠름
- 실무에서 심사 시스템이 한 번에 수십 건을 요청하는 경우가 많음

In [ ]:
# 여기에 코드를 작성하세요


## 3. 모델 메타 정보 API

### WHY `/model/info` 엔드포인트가 필요한가?

운영 환경에서 다음 질문에 답할 수 있어야 합니다:
- "지금 서빙 중인 모델이 어떤 버전인가?"
- "어떤 피처를 사용하고 있는가?"
- "승인 임계값은 얼마인가?"

이 정보를 API로 노출하면:
- CI/CD 파이프라인에서 배포 후 자동 검증 가능
- 모니터링 대시보드에서 실시간 모델 상태 표시
- 디버깅 시 "어떤 모델이 이 결과를 냈는지" 추적 가능

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


## 정리

### 이번 세션에서 배운 것

| 주제 | 핵심 내용 |
|------|----------|
| Lifespan | `@asynccontextmanager`로 서버 시작/종료를 하나의 함수에서 관리 |
| 배치 예측 | `List[LoanRequest]` → `List[LoanResponse]`로 여러 건을 한 번에 처리 |
| 모델 정보 | `/model/info`로 운영 중 모델 상태 확인 (버전, 피처, 임계값) |
| 요청 추적 | `uuid4()` + ISO 8601 timestamp로 각 요청을 고유하게 식별 |

### 실무에서의 활용
- **배치 API**: 대량 심사, 정기 배치 처리에 필수
- **모델 정보 API**: CI/CD 파이프라인에서 배포 후 검증, 모니터링 대시보드
- **요청 추적**: 로그 분석, 문제 추적, 감사(audit) 목적

